# personal_compare – Personalisierungs-Experimente (alles in einem Notebook)

Dieses Notebook führt **alle** noch offenen Experimente von Kapitel 4 in der richtigen Reihenfolge aus und erzeugt am Ende die Tabellen.
Es läuft auf einer **CPU**: Weil der Backbone eingefroren ist, werden seine Merkmale (512 Zahlen pro Bild) nur **einmal** pro Session berechnet und gespeichert (*Feature-Cache*). Danach wird nur der kleine Kopf trainiert – ein Lauf dauert dann Sekunden.

| Schritt | Experiment | Wozu? | Läufe | Dauer (CPU, geschätzt) |
|---|---|---|---|---|
| 2 | **E0** Basismodelle | beste Hyperparameter-Konfiguration für `norm_subject` und `norm_random` trainieren und speichern | 2 × 5 Seeds | 5–15 min |
| 3 | Feature-Cache | Backbone-Merkmale aller 18 Sessions für beide Basismodelle | 36 Dateien | 15–40 min (einmalig) |
| 4 | **E1** Alle Sessions | Tabelle 4.13: 18 Sessions × 2 Basismodelle | 36 | wenige Minuten |
| 5 | **E1b** Bereinigte Labels | Gen 4 ohne WAIT-Frames vs. Original | 9 | < 1 min |
| 6 | **E2** Data Loading | Random / Sequential / Two-Video auf Session 21 und 22, 3 Seeds | 18 | wenige Minuten |
| 7 | **E3** Session-Design | Sequential-Split für alle 18 Sessions → fairer Design-Vergleich + Fehler pro Bildschirmbereich | 18 | wenige Minuten |
| 8 | Auswertung | alle Tabellen, Tests und Grafiken | – | Sekunden |

**Wichtig**
* Jeder fertige Lauf wird sofort in `results/personal/personal_runs.jsonl` gespeichert. Bricht etwas ab, einfach die Zelle erneut starten – fertige Läufe werden übersprungen.
* Gemeinsames Protokoll aller Läufe: Kopf (FC) trainierbar, Backbone inkl. BatchNorm eingefroren, Sigmoid, L1-Loss, AdamW (lr = 1e-4, weight decay = 1e-5), Batch 32, Early Stopping (Patience 3) auf der **Validierung**, Testfehler am besten Validierungs-Epoch, feste Seeds. Keine Bild-Augmentierung (Feature-Cache).
* Metrik: Diagonalfehler $E_\text{diag}=100\cdot\text{RMSE}_\text{eukl.}/\sqrt2$ (Gl. 4.1, `metrics.py`).
* Am Ende (Abschnitt 9) die Ergebnisse mit Git hochladen.

## 1. Setup und Prüfung der Voraussetzungen

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import torch
import personal_exp as pe
import train_base
import session_design as sd
import spatial_analysis as sa

torch.set_num_threads(max(1, os.cpu_count() or 1))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 4            # bei Problemen mit DataLoader-Prozessen auf 0 setzen
print("Projekt:", ROOT, "| Gerät:", DEVICE, "| CPU-Threads:", torch.get_num_threads())

# Voraussetzungen prüfen: Labels und Bilder jeder Session
rows = []
for s in pe.ALL_SESSIONS:
    d = pe.SESSION_ROOT / s
    img = d / "images2" if (d / "images2").is_dir() else d / "images"
    n_lab = len(pd.read_csv(d / "norm_labels.csv")) if (d / "norm_labels.csv").exists() else 0
    n_img = len(list(img.glob("*.jpg"))) if img.is_dir() else 0
    rows.append(dict(session=s, labels=n_lab, images=n_img, folder=img.name, ok=n_lab > 0 and n_img >= n_lab))
check = pd.DataFrame(rows)
display(check)
assert check.ok.all(), "Für einige Sessions fehlen Bilder oder Labels (siehe Tabelle)."

## 2. E0 – Basismodelle trainieren und speichern

**Warum?** Die Hyperparameter-Experimente (`exp.py`) haben keine Modelle gespeichert. Für die Personalisierung brauchen wir aber je ein Basismodell für `norm_subject` und `norm_random`.

**Wie?** Die ausgewählte Konfiguration (FC only, Sigmoid, L1, lr 1e-4, Batch 32, 1000/200 zufällige GazeCapture-Bilder mit Seed 0, Validierung 200 Bilder eines eigenen Probanden) wird mit 5 Seeds trainiert. Gespeichert wird der Lauf mit dem **kleinsten Validierungsfehler** (das Testset wird für die Auswahl nicht benutzt) → `models/base/base_<split>.pt` + `.json`.

**Erwartung:** Testfehler ≈ Konstant-Baseline (≈ 28,5 % bzw. ≈ 27,7 %), wie in Abschnitt 4.2 des Kapitels. Vorhandene Basismodelle werden nicht überschrieben.

In [ ]:
base_meta = train_base.train_all(splits=("norm_subject", "norm_random"), seeds=(0, 1, 2, 3, 4))
pd.DataFrame([{"split": k, "selected_seed": v["selected_seed"],
               **{f"seed{r['seed']}_val": round(r["val_error"], 2) for r in v["runs"]},
               "const_baseline": round(v["baseline_same_subset"]["constant"], 2)} for k, v in base_meta.items()])

## 3. Feature-Cache für alle Sessions

**Warum?** Der eingefrorene Backbone liefert für ein Bild in jeder Epoche dieselben Merkmale. Sie werden hier **einmal** berechnet (`cache/personal/<base>_<session>.pt`). Das ist der einzige rechenintensive Schritt auf der CPU.

In [ ]:
t0 = time.perf_counter()
for base in ("norm_subject", "norm_random"):
    for s in pe.ALL_SESSIONS:
        pe.session_features(s, base, DEVICE, num_workers=NUM_WORKERS)
print(f"fertig nach {(time.perf_counter() - t0) / 60:.1f} min")

## 4. E1 – Alle 18 Sessions mit beiden Basismodellen

**Warum?** Vervollständigt Tabelle 4.13: für die Sessions 19–27 fehlt `norm_random`, und alle alten Werte stammen aus Läufen ohne festen Seed. Zufälliger Split 70/15/15 (Seed 42), Original-Labels. Die Testvorhersagen werden gespeichert (für Abschnitt 8.5).

**Erwartung:** deutlich unter der Konstant-Baseline (alte Werte: Mittel ≈ 20 % vs. ≈ 30 %); beide Basismodelle ähnlich.

In [ ]:
e1 = pe.run_series(sessions=["all"], bases=["norm_subject", "norm_random"], labels=["original"],
                   split="random", seeds=[42], num_workers=NUM_WORKERS)

## 5. E1b – Gen 4 mit korrekt bereinigten Labels

**Warum?** Im alten Vergleich waren die „bereinigten“ Dateien identisch mit den Originalen. Jetzt fehlen in `norm_labels_clean.csv` genau die WAIT-Frames (10 %), in denen der Blick den neuen Punkt noch sucht. Vergleich mit E1 (gleicher Seed, gleiches Basismodell).

In [ ]:
e1b = pe.run_series(sessions=["gen4"], bases=["norm_subject"], labels=["clean"],
                    split="random", seeds=[42], num_workers=NUM_WORKERS)

## 6. E2 – Data Loading Strategies (Session 21 und 22)

**Warum?** Beim zufälligen Split liegen fast identische Nachbarframes in Training und Test (Datenleckage) → zu optimistisch.

| Strategie | Training | Validierung | Test |
|---|---|---|---|
| Random | 70 % zufällig | 15 % zufällig | 15 % zufällig (gleiche Session) |
| Sequential | erste 70 % | nächste 15 % (ohne die ersten 1 s) | letzte 15 % (ohne die ersten 1 s) |
| Two-Video | 80 % von Session A | 20 % von Session A | **alle** Frames von Session B |

Sessions 21 und 22: gleiche Person, gleicher Tag, ohne Brille, ruhiger Hintergrund, Ubuntu; sie unterscheiden sich nur in der Dauer (120 s / 90 s). Basismodell `norm_subject`, Seeds 42/43/44.

**Erwartung:** Random < Sequential ≤ Two-Video (Fehler).

In [ ]:
seeds = [42, 43, 44]
e2 = []
e2 += pe.run_series(sessions=["21", "22"], split="random", seeds=seeds, num_workers=NUM_WORKERS)
e2 += pe.run_series(sessions=["21", "22"], split="sequential", seeds=seeds, gap_sec=1.0, num_workers=NUM_WORKERS)
e2 += pe.run_series(split="two_videos", pairs=[("21", "22"), ("22", "21")], seeds=seeds, num_workers=NUM_WORKERS)

## 7. E3 – Session-Design fair vergleichen (Sequential-Split, alle Sessions)

**Warum?** Der zufällige Split bevorzugt die kontinuierlichen Gen-1-Videos (Nachbarframes in Training und Test). Mit dem Sequential-Split wird der Vergleich Standard / Random Jumps / Corner-weighted fairer. Die gespeicherten Vorhersagen liefern die Fehler im Zentrum, an den Rändern und in den Ecken.

In [ ]:
e3 = pe.run_series(sessions=["all"], bases=["norm_subject"], split="sequential", seeds=[42],
                   gap_sec=1.0, num_workers=NUM_WORKERS)

## 8. Auswertung
Alle Werte kommen aus `results/personal/personal_runs.jsonl` (nur Läufe mit Feature-Cache).

In [ ]:
runs = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in pe.load_runs()])
if "features" in runs:
    runs = runs[runs.features == "cached"].copy()
GEN = {s: g.replace("gen", "Gen ") for g, ss in pe.SESSIONS_GEN.items() for s in ss}
runs["generation"] = runs.session.map(GEN)

def ci98(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (x.mean() if len(x) else np.nan, np.nan, np.nan)
    m = x.mean(); h = stats.t.ppf(0.99, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return m, m - h, m + h

fmt = lambda t: f"{t[0]:.2f} [{t[1]:.2f}, {t[2]:.2f}]"
print(len(runs), "Läufe")

### 8.1 Tabelle 4.13 – alle Sessions, beide Basismodelle
Val. = bester Validierungsfehler, Test = Testfehler am besten Validierungs-Epoch, t = Dauer eines Laufs (nur Kopf-Training inkl. Auswertung je Epoche, ohne Feature-Berechnung), Const. = Konstant-Baseline (Mittel der Trainingsziele, auf den Testframes ausgewertet), Base = unverändertes Basismodell auf denselben Testframes.

In [ ]:
e1r = runs[(runs.split == "random") & (runs.labels == "original") & (runs.seed == 42)]
tab = e1r.pivot_table(index=["generation", "session"], columns="base",
                      values=["val_error", "test_error", "minutes", "const_baseline", "epoch0_test_error"])
display(tab.round(2))

grp = []
for name, ss in [("Gen 1", pe.SESSIONS_GEN["gen1"]), ("Gen 2", pe.SESSIONS_GEN["gen2"]),
                 ("Gen 4", pe.SESSIONS_GEN["gen4"]), ("Alle 18", pe.ALL_SESSIONS)]:
    for base in ("norm_subject", "norm_random"):
        d = e1r[(e1r.base == base) & e1r.session.isin(ss)]
        grp.append({"Gruppe": name, "Basis": base, "n": len(d),
                    "Val": fmt(ci98(d.val_error)), "Test": fmt(ci98(d.test_error)),
                    "t (min)": fmt(ci98(d.minutes)), "Const": fmt(ci98(d.const_baseline)),
                    "Basismodell": fmt(ci98(d.epoch0_test_error))})
display(pd.DataFrame(grp))
pe.latex_sessions(e1r)          # LaTeX-Zeilen für das Kapitel

### 8.2 Personalisierung vs. Baselines, Basismodelle im Vergleich

In [ ]:
for base in ("norm_subject", "norm_random"):
    d = e1r[e1r.base == base].sort_values("session")
    print(f"{base}: Test {d.test_error.mean():.2f}% vs. Konstant {d.const_baseline.mean():.2f}% "
          f"vs. Basismodell {d.epoch0_test_error.mean():.2f}% | Wilcoxon Test<Konstant p={stats.wilcoxon(d.test_error, d.const_baseline).pvalue:.2g} "
          f"| Sessions ohne Gewinn: {d[d.test_error >= d.const_baseline].session.tolist()}")
a = e1r[e1r.base == "norm_subject"].set_index("session").test_error
b = e1r[e1r.base == "norm_random"].set_index("session").test_error
common = a.index.intersection(b.index)
print("norm_subject vs. norm_random (gepaart):", stats.wilcoxon(a[common], b[common]))
print("Generationen (Kruskal-Wallis, norm_subject):",
      stats.kruskal(*[e1r[(e1r.base == "norm_subject") & (e1r.generation == g)].test_error for g in ("Gen 1", "Gen 2", "Gen 4")]))

S = sorted(e1r.session.unique()); x = np.arange(len(S)); w = 0.27
v = lambda base, col: e1r[e1r.base == base].set_index("session").reindex(S)[col]
fig, ax = plt.subplots(figsize=(11, 4.2), constrained_layout=True)
ax.bar(x - w, v("norm_subject", "const_baseline"), w, color="#9e9e9e", label="Constant baseline")
ax.bar(x, v("norm_subject", "test_error"), w, color="#1f77b4", label="Personalized (norm_subject)")
ax.bar(x + w, v("norm_random", "test_error"), w, color="#ff7f0e", label="Personalized (norm_random)")
ax.set_xticks(x, S); ax.set_xlabel("Session"); ax.set_ylabel("Diagonal error (%)"); ax.grid(axis="y", alpha=0.3)
fig.legend(loc="outside lower center", ncol=3, fontsize=8, frameon=False)
Path("results/figures").mkdir(parents=True, exist_ok=True)
fig.savefig("results/figures/fig_sessions_baseline.png", dpi=200); plt.show()

### 8.3 E1b – Original vs. bereinigt (Gen 4)

In [ ]:
sel = (runs.split == "random") & (runs.base == "norm_subject") & (runs.seed == 42)
o = runs[sel & (runs.labels == "original")].set_index("session").test_error
c = runs[sel & (runs.labels == "clean")].set_index("session").test_error
g4 = pd.DataFrame({"original": o, "clean": c}).dropna()
g4["delta"] = g4.clean - g4.original
display(g4.round(2))
if len(g4) > 1:
    print("Mittel:", g4.mean().round(2).to_dict(), "| Wilcoxon:", stats.wilcoxon(g4.clean, g4.original))

### 8.4 E2 – Data Loading Strategies
Pro Session und Strategie: Mittelwert ± Standardabweichung über 3 Seeds. Die Testmengen sind verschieden groß; deshalb stehen Konstant-Baseline und Basismodell auf **denselben** Testframes daneben.

In [ ]:
e2r = runs[runs.session.isin(["21", "22"]) & (runs.base == "norm_subject") & (runs.labels == "original")
           & runs.seed.isin([42, 43, 44])]
e2t = (e2r.groupby(["split", "session", "test_session"])
          .agg(test_mean=("test_error", "mean"), test_sd=("test_error", "std"), val_mean=("val_error", "mean"),
               const=("const_baseline", "mean"), base_model=("epoch0_test_error", "mean"),
               n_train=("n_train", "first"), n_test=("n_test", "first"), runs=("seed", "count")))
display(e2t.round(2))
print(e2r.groupby("split").test_error.mean().round(2))

### 8.5 E3 – Session-Design: Sequential-Split und Fehler pro Bildschirmbereich
Bereiche wie in Kapitel 4.3.1 (Gitterpunkte, r = 0,15). Fehler = mittlerer euklidischer Fehler (normierte Einheiten), über alle Testframes einer Generation gepoolt.

In [ ]:
e3r = runs[(runs.split == "sequential") & (runs.base == "norm_subject") & (runs.labels == "original") & (runs.seed == 42)]
cmp = pd.DataFrame({"random": e1r[e1r.base == "norm_subject"].set_index("session").test_error,
                    "sequential": e3r.set_index("session").test_error,
                    "const_seq": e3r.set_index("session").const_baseline})
cmp["generation"] = cmp.index.map(GEN)
display(cmp.groupby("generation").mean().round(2))
display(cmp.round(2))

region = []
for _, r in e3r.iterrows():
    f = Path("results/personal/predictions") / f"pred_{r.session}-{r.session}_norm_subject_original_sequential_standard_s42.csv"
    if not f.exists(): continue
    gt, pr = sa.load_predictions(f)
    err = np.linalg.norm(gt - pr, axis=1)
    for name, pts in (("centre", sa.CENTER), ("edge", sa.EDGES), ("corner", sa.CORNERS)):
        m = np.zeros(len(gt), bool)
        for p in pts: m |= np.linalg.norm(gt - np.array(p), axis=1) <= 0.15
        region += [{"generation": GEN[r.session], "region": name, "error": e} for e in err[m]]
reg = pd.DataFrame(region)
if len(reg):
    display(reg.groupby(["generation", "region"]).error.agg(["mean", "count"]).unstack().round(3))

### 8.6 Räumliche Abdeckung und Übergangs-Samples (ohne Training)

In [ ]:
design = sd.analyse()
display(design.round(3))
display(design.groupby("generation").mean(numeric_only=True).round(3))

## 9. Ergebnisse hochladen
Im Terminal (Projektordner) auf diesem Rechner:
```bash
git add results/personal/personal_runs.jsonl results/personal/predictions models/base/*.json results/figures/fig_sessions_baseline.png
git commit -m "Personalisierung: E0-E3 (Feature-Cache)"
git pull --rebase
git push
```
Die großen Dateien `models/base/*.pt` und `cache/` werden **nicht** hochgeladen (nicht nötig für die Auswertung).